In [5]:
import os
os.chdir('../')
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainedModelConfig:
    checkpoint_dir  : Path
    latest_model_dir: Path
    best_model_dir  : Path
    run_info_dir    : Path

@dataclass(frozen=True)
class OnnxConfig:
    root_dir:               Path
    trained_model:          TrainedModelConfig
    onnx_model_dir:         Path
    onnx_int8_model_dir:    Path
    image_size:             int

In [3]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    # ── private helpers ───────────────────────────────────────
    def _model_slug(self) -> str:
        p = self.params.prepare_base_model_params
        return f"{p.model_name}_{p.encoder}"

    def _get_trained_model_config(self, train_root: Path) -> TrainedModelConfig:
        slug = self._model_slug()
        d    = train_root / slug
        
        return TrainedModelConfig(
            checkpoint_dir  = d / "checkpoints",
            latest_model_dir= d / "model.pth",
            best_model_dir  = d / "best_model.pth",
            run_info_dir    = d / "run_info.json"
        )
    
    # ── public ───────────────────────────────────────────────
    def get_onnx_config(self) -> OnnxConfig:
        onnx_root  = Path(self.config.onnx_config.root_dir) 
        train_root = Path(self.config.training_config.root_dir)
        slug       = self._model_slug()

        return OnnxConfig(
            root_dir        = onnx_root,
            trained_model   = self._get_trained_model_config(train_root),
            onnx_model_dir      = onnx_root / slug / "model.onnx",
            onnx_int8_model_dir = onnx_root / slug / "model_int8.onnx",
            image_size      = self.params.onnx_params.image_size
        )

In [4]:
import sys, json, torch
import segmentation_models_pytorch as smp
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType

from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException

MODEL_MAP = {
    "unet":       smp.Unet,
    "unetpp":     smp.UnetPlusPlus,
    "deeplabv3":  smp.DeepLabV3,
    "deeplabv3p": smp.DeepLabV3Plus,
    "manet":      smp.MAnet,
}

class Onnx:
    def __init__(self, config: OnnxConfig):
        self.config = config
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model  = self._load_model()
        
    def _load_model(self) -> torch.nn.Module:
        try:
            run_info = json.loads(
                Path(self.config.trained_model.run_info_dir).read_text()
            )
            model_name = run_info["model_name"].lower()
            if model_name not in MODEL_MAP:
                raise ValueError(f"Model '{model_name}' not in MODEL_MAP")
            
            model = MODEL_MAP[model_name](
                encoder_name    = run_info["encoder"],
                encoder_weights = None,
                classes         = 1,
                activation      = None,
            )
            model.load_state_dict(torch.load(
                self.config.trained_model.best_model_dir,
                map_location = self.device,
                weights_only = True,
            ))
            logging.info(f"Loaded {model_name} + {run_info['encoder']} from best_model.pth")
            return model.eval().to(self.device)
        
        except Exception as e:
            raise CustomException(e, sys)
    
    def export_onnx(self):
        try: 
            self.config.onnx_model_dir.parent.mkdir(parents=True, exist_ok=True)
            dummy_input = torch.randn(
                1, 3, self.config.image_size, self.config.image_size
            ).to(self.device)
            
            torch.onnx.export(
                self.model, dummy_input,
                str(self.config.onnx_model_dir),
                opset_version = 17,
                input_names   = ["input"],
                output_names  = ["output"],
                dynamic_axes  = {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
            )
            logging.info(f"ONNX exported -> {self.config.onnx_model_dir}")
            
        except Exception as e:
            raise CustomException(e, sys)
    
    def quantize(self):
        try: 
            quantize_dynamic(
                model_input = str(self.config.onnx_model_dir),
                model_output= str(self.config.onnx_int8_model_dir),
                weight_type = QuantType.QUInt8
            )
            logging.info(f"INT8 model -> {self.config.onnx_int8_model_dir}")
            
        except Exception as e:
            raise CustomException(e, sys)

d:\Deep_Learning_Object_Detection\MLOPs\pneumonia-segmentation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
try:
    config = ConfigurationManger()
    onnx_config = config.get_onnx_config()
    
    onnx = Onnx(config=onnx_config)
    onnx.export_onnx()
    onnx.quantize()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-06 00:15:32,316: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-06 00:15:32,320: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-06 00:15:32,322: INFO: common: created directory at: artifacts]
[2026-04-06 00:15:32,706: INFO: 370003583: Loaded unetpp + efficientnet-b3 from best_model.pth]


d:\Deep_Learning_Object_Detection\MLOPs\pneumonia-segmentation\.venv\Lib\site-packages\torch\onnx\_internal\jit_utils.py:309: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\jit\passes\onnx\constant_fold.cpp:180.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
d:\Deep_Learning_Object_Detection\MLOPs\pneumonia-segmentation\.venv\Lib\site-packages\torch\onnx\utils.py:691: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\jit\passes\onnx\constant_fold.cpp:180.)
  _C._jit_pass_onnx_graph_shape_type_inference(
d:\Deep_Learning_Object_Detection\MLOPs\pneumonia-segmentation\.venv\Lib\site-packages\torch\onnx\utils.py:1161: UserWarning: 

[2026-04-06 00:15:36,471: INFO: 370003583: ONNX exported -> artifacts\onnx\unetpp_efficientnet-b3\model.onnx]
[2026-04-06 00:15:37,032: WARNING: quantize: Please consider to run pre-processing before quantization. Refer to example: https://github.com/microsoft/onnxruntime-inference-examples/blob/main/quantization/image_classification/cpu/ReadMe.md ]
[2026-04-06 00:15:39,007: WARNING: onnx_quantizer: Inference failed or unsupported type to quantize for tensor '/encoder/_conv_stem/static_padding/Slice_output_0', type is tensor_type {
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.]
[2026-04-06 00:15:39,010: INFO: onnx_quantizer: Quantization parameters for tensor:"/encoder/_conv_stem/static_padding/Pad_output_0" not specified]
[2026-04-06 00:15:39,013: WARNING: onnx_quantizer: Inference failed or unsupported type to quantize for tensor '/encoder/_blocks.0/_depthwise_conv/static_padding/Slice_output_0', type is tensor_type {
  elem_ty

In [6]:
import segmentation_models_pytorch as smp
import torch
import json, time
import onnxruntime as ort
from pathlib import Path

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
run_info = json.loads(Path('artifacts/training/manet_resnet34/run_info.json').read_text())

model = smp.MAnet(
    encoder_name    = run_info['encoder'],
    encoder_weights = None,
    classes         = 1,
    activation      = None,
)
model.load_state_dict(torch.load(
    'artifacts/training/manet_resnet34/best_model.pth',
    map_location = DEVICE,
    weights_only = True,
))
model.eval().to(DEVICE)
print('Model loaded')

Model loaded


In [7]:
import numpy as np
ONNX_PATH  = Path('artifacts/onnx/manet_resnet34/model.onnx')
IMAGE_SIZE = 256
N_RUNS     = 50
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

dummy_np    = np.random.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).astype(np.float32)
dummy_torch = torch.from_numpy(dummy_np).to(DEVICE)

def measure(fn, n_runs=N_RUNS, warmup=5):
    for _ in range(warmup):
        fn()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_runs):
        fn()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_runs * 1000  # ms

In [8]:
results = {}

# 1. PyTorch CUDA
with torch.no_grad():
    results['pytorch_cuda'] = measure(lambda: model(dummy_torch))
print(f"PyTorch CUDA: {results['pytorch_cuda']:.2f} ms")

# 2. ONNX CPU
sess_cpu = ort.InferenceSession(
    str(ONNX_PATH),
    providers=['CUDAExecutionProvider']
)
results['onnx_cpu'] = measure(lambda: sess_cpu.run(None, {'input': dummy_np}))
print(f"ONNX CPU:     {results['onnx_cpu']:.2f} ms")

model_cpu = model.cpu()
dummy_cpu = dummy_torch.cpu()
with torch.no_grad():
    results['pytorch_cpu'] = measure(lambda: model_cpu(dummy_cpu))
print(f"PyTorch CPU:  {results['pytorch_cpu']:.2f} ms")

PyTorch CUDA: 12.81 ms
ONNX CPU:     7.21 ms
PyTorch CPU:  93.87 ms
